# 03 - curated h5ad の `.X` 前処理状態を診断する

このノートブックは、02 で作成した curated h5ad の `.X` が **どの前処理状態に近いか** を
推定するための診断ノートです。`pandas / numpy / scanpy / anndata / scipy / matplotlib / pathlib`
だけで動く単純な構成にしています。

## このノートブックの方針

- **データを正規化しない。** normalize / log1p / scale などは一切行わない。
- **QC フィルタリングもしない。** total_counts や pct_counts_mt などでのセル除去はしない。
- `.X` の値分布から、raw count / log normalized / TPM / scaled などを **推定** する。
- 値だけから前処理状態を完全に断定することはできない（あくまで診断）。
- ここでの判定は **04 の merge 方針を決めるための診断** である。
- raw count ではないデータに、通常の count-based QC や scVI をそのまま適用しない。

## 推定したい状態

```
raw_count_like        : 生カウント（非負・整数・max が大きい）
processed_count_like  : SoupX 補正カウントなど（カウント様だが非整数を含む）
log_normalized_like   : log1p 正規化後（非負・非整数・max が小さい）
cpm_tpm_like          : CPM/TPM（row sum が 1e4/1e5/1e6 付近の定数）
scaled_zscore_like    : z-score scale 後（負の値を含む / gene std ~ 1）
unknown               : 上記に明確に当てはまらない
```

## QC について

- QC 本番は **04 で merge 対象を決めた後** に行う。
- 通常の `total_counts`, `n_genes_by_counts`, `pct_counts_mt` ベースの QC が適しているのは
  **raw_count_like のデータだけ** である。
- `log_normalized_like` や `scaled_zscore_like` のデータでは count-based QC の解釈が変わる。
- したがって、この 03 では preprocessing state の確認を優先する。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib
matplotlib.use("Agg")  # 非対話バックエンド（保存優先）
import matplotlib.pyplot as plt

ROOT = Path("../../").resolve()
CURATED_DIR = ROOT / "data" / "curated_h5ad"
OUT_DIR = ROOT / "results" / "preprocessing_state"
PLOT_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("curated dir:", CURATED_DIR)
print("out dir    :", OUT_DIR)
print("plot dir   :", PLOT_DIR)

# サンプリング上限（大規模データで重くならないように）
MAX_CELLS = 20000   # 行（細胞）サンプリング上限
MAX_VALS = 200000   # 非ゼロ値サンプリング上限
MAX_GENES = 4000    # gene 統計用の遺伝子サンプリング上限
MAX_PLOT = 50000    # プロット用サンプリング上限
RNG = np.random.default_rng(0)

## 入力 curated h5ad の一覧（CSV は除外）

In [ ]:
SKIP_FILES = {"original_obs_metadata_by_cell.csv", "curation_summary.csv"}

curated_files = sorted(p for p in CURATED_DIR.glob("*.h5ad"))
print(f"{len(curated_files)} curated h5ad files:\n")
for p in curated_files:
    print(p.name)

## `.X` から診断指標を計算する関数

sparse をそのまま扱い、全要素を dense 化しない。値分布は非ゼロ値を中心にサンプリングする。

In [ ]:
def _to_1d(a):
    return np.asarray(a).ravel()


def _percentiles(vals, qs):
    if vals.size == 0:
        return {q: float("nan") for q in qs}
    p = np.percentile(vals, qs)
    return {q: float(v) for q, v in zip(qs, np.atleast_1d(p))}


def compute_metrics(adata):
    """adata.X から前処理状態の診断指標を計算して dict と plot 用配列を返す。"""
    X = adata.X
    n_cells, n_genes = adata.shape
    is_sp = sp.issparse(X)

    # --- 行（細胞）サンプリング ---
    if n_cells > MAX_CELLS:
        ridx = np.sort(RNG.choice(n_cells, size=MAX_CELLS, replace=False))
        Xs = X[ridx]
    else:
        Xs = X
    n_sampled_cells = Xs.shape[0]

    # --- zero / negative fraction（zero は nnz から） ---
    total = Xs.shape[0] * Xs.shape[1]
    if is_sp:
        Xs = Xs.tocsr()
        nnz = Xs.nnz
        data = Xs.data
    else:
        Xs = np.asarray(Xs)
        data = Xs[Xs != 0]
        nnz = data.size
    fraction_zero = 1.0 - (nnz / total) if total else float("nan")

    xmin = float(Xs.min()) if total else float("nan")
    xmax = float(Xs.max()) if total else float("nan")
    xmean = float(Xs.sum() / total) if total else float("nan")

    # --- 非ゼロ値サンプリング ---
    nz = data[data != 0]
    if nz.size > MAX_VALS:
        vals = RNG.choice(nz, size=MAX_VALS, replace=False)
    else:
        vals = nz
    vals = np.asarray(vals, dtype=float)

    if vals.size:
        fraction_negative = float((vals < 0).mean())
        if xmin < 0:  # 最小値が負なら下駄をはかせる
            fraction_negative = max(fraction_negative, 1e-6)
        fraction_integer_like = float(np.isclose(vals, np.round(vals), atol=1e-6).mean())
        pct = _percentiles(vals, [50, 90, 99])
    else:
        fraction_negative = 0.0
        fraction_integer_like = float("nan")
        pct = {50: float("nan"), 90: float("nan"), 99: float("nan")}

    # --- row sum / n_genes_by_cell（サンプリング済み Xs で） ---
    row_sum = _to_1d(Xs.sum(axis=1))
    n_genes_by_cell = _to_1d((Xs > 0).sum(axis=1))

    rs_med = float(np.median(row_sum)) if row_sum.size else float("nan")
    rs_mean = float(np.mean(row_sum)) if row_sum.size else float("nan")
    rs_std = float(np.std(row_sum)) if row_sum.size else float("nan")
    rs_cv = (rs_std / rs_mean) if rs_mean not in (0, float("nan")) and rs_mean != 0 else float("nan")
    rs_p = _percentiles(row_sum, [10, 90])
    ng_med = float(np.median(n_genes_by_cell)) if n_genes_by_cell.size else float("nan")
    ng_p = _percentiles(n_genes_by_cell, [10, 90])

    # --- gene mean / std（遺伝子サンプリング可） ---
    if n_genes > MAX_GENES:
        gidx = np.sort(RNG.choice(n_genes, size=MAX_GENES, replace=False))
        Xg = Xs[:, gidx]
    else:
        Xg = Xs
    if is_sp:
        gmean = _to_1d(Xg.mean(axis=0))
        gmeansq = _to_1d(Xg.multiply(Xg).mean(axis=0))
    else:
        gmean = _to_1d(np.asarray(Xg).mean(axis=0))
        gmeansq = _to_1d((np.asarray(Xg) ** 2).mean(axis=0))
    gstd = np.sqrt(np.maximum(gmeansq - gmean ** 2, 0.0))

    metrics = {
        "n_cells": int(n_cells),
        "n_genes": int(n_genes),
        "X_format": type(X).__name__,
        "has_raw": adata.raw is not None,
        "layers": ", ".join(adata.layers.keys()),
        "min": xmin,
        "max": xmax,
        "mean": xmean,
        "p50": pct[50],
        "p90": pct[90],
        "p99": pct[99],
        "fraction_negative": fraction_negative,
        "fraction_zero": float(fraction_zero),
        "fraction_integer_like": fraction_integer_like,
        "row_sum_median": rs_med,
        "row_sum_p10": rs_p[10],
        "row_sum_p90": rs_p[90],
        "row_sum_cv": float(rs_cv),
        "n_genes_by_cell_median": ng_med,
        "n_genes_by_cell_p10": ng_p[10],
        "n_genes_by_cell_p90": ng_p[90],
        "gene_mean_median": float(np.median(gmean)) if gmean.size else float("nan"),
        "gene_mean_abs_median": float(np.median(np.abs(gmean))) if gmean.size else float("nan"),
        "gene_std_median": float(np.median(gstd)) if gstd.size else float("nan"),
    }
    plot_data = {
        "values": vals[:MAX_PLOT],
        "row_sums": row_sum if row_sum.size <= MAX_PLOT else RNG.choice(row_sum, MAX_PLOT, replace=False),
        "n_genes": n_genes_by_cell if n_genes_by_cell.size <= MAX_PLOT else RNG.choice(n_genes_by_cell, MAX_PLOT, replace=False),
    }
    return metrics, plot_data

## 推定ルール（`guess_preprocessing`）

値だけから断定はできないため、ヒューリスティックで状態を当てる。

In [ ]:
INCLUDE_FOR_COUNT_MERGE = {
    "raw_count_like": True,
    "processed_count_like": False,  # 後で人間が判断する
    "log_normalized_like": False,
    "cpm_tpm_like": False,
    "scaled_zscore_like": False,
    "unknown": False,
}


def guess_preprocessing(m):
    fn = m["fraction_negative"]
    fil = m["fraction_integer_like"]
    mx = m["max"]
    p99 = m["p99"]
    rs_med = m["row_sum_median"]
    rs_cv = m["row_sum_cv"]
    gstd = m["gene_std_median"]
    gabs = m["gene_mean_abs_median"]

    fil = 0.0 if fil != fil else fil  # NaN -> 0

    # scaled / z-score: 負の値を含む、または gene mean ~ 0 かつ gene std ~ 1
    if fn > 0.01 or (gabs == gabs and gstd == gstd and gabs < 0.1 and 0.5 < gstd < 2.0):
        conf = "high" if fn > 0.01 else "medium"
        return "scaled_zscore_like", conf, f"frac_neg={fn:.3f}, gene_std_median={gstd:.2f}, gene_mean_abs_median={gabs:.3f}"

    # raw count: 非負・整数・max 大・row_sum のばらつき大
    if fn == 0 and fil > 0.99 and mx > 20 and rs_cv == rs_cv and rs_cv > 0.2:
        return "raw_count_like", "high", f"integer_frac={fil:.3f}, max={mx:.0f}, row_sum_cv={rs_cv:.2f}"

    # cpm/tpm: row_sum が 1e4/1e5/1e6 付近の定数 & cv 小
    for target in (1e4, 1e5, 1e6):
        if (fn == 0 and fil < 0.8 and rs_cv == rs_cv and rs_cv < 0.05
                and rs_med == rs_med and abs(rs_med - target) / target < 0.1):
            return "cpm_tpm_like", "high", f"row_sum_median~{rs_med:.0f}, row_sum_cv={rs_cv:.3f}"

    # log normalized: 非負・非整数・max 小・p99 小
    if fn == 0 and fil < 0.9 and mx == mx and mx <= 20 and p99 == p99 and p99 < 12:
        return "log_normalized_like", "medium", f"non-integer, max={mx:.2f}, p99={p99:.2f}"

    # processed count: 非負・非整数だが max/row_sum が count 様（SoupX corrected など）
    if fn == 0 and fil < 0.8 and mx == mx and mx > 20:
        return "processed_count_like", "low", f"non-integer (frac_int={fil:.2f}) but count-like max={mx:.0f}"

    return "unknown", "low", "no clear rule matched"


def finalize_guess(m):
    guess, conf, note = guess_preprocessing(m)
    m["guess_preprocessing"] = guess
    m["confidence"] = conf
    m["include_for_count_merge"] = INCLUDE_FOR_COUNT_MERGE.get(guess, False)
    m["notes"] = note
    return m

## 全 curated h5ad を診断

In [ ]:
rows = []
plot_store = {}

for p in curated_files:
    adata = sc.read_h5ad(p)
    dataset_id = p.stem
    source_accession = adata.obs["source_accession"].iloc[0] if "source_accession" in adata.obs.columns else dataset_id.split("_")[0]

    m, plot_data = compute_metrics(adata)
    m = finalize_guess(m)
    m["dataset_id"] = dataset_id
    m["source_accession"] = source_accession
    rows.append(m)
    plot_store[(source_accession, dataset_id)] = plot_data

    print(f"{source_accession:12s} {dataset_id:55s} -> {m['guess_preprocessing']:22s} ({m['confidence']})")

col_order = [
    "dataset_id", "source_accession", "n_cells", "n_genes",
    "X_format", "has_raw", "layers",
    "min", "max", "mean", "p50", "p90", "p99",
    "fraction_negative", "fraction_zero", "fraction_integer_like",
    "row_sum_median", "row_sum_p10", "row_sum_p90", "row_sum_cv",
    "n_genes_by_cell_median", "n_genes_by_cell_p10", "n_genes_by_cell_p90",
    "gene_mean_median", "gene_mean_abs_median", "gene_std_median",
    "guess_preprocessing", "confidence", "include_for_count_merge", "notes",
]
summary = pd.DataFrame(rows)
summary = summary[[c for c in col_order if c in summary.columns]]
print(f"\ndiagnosed {len(summary)} datasets")

## 可視化（非ゼロ値 / row sum / n_genes_by_cell のヒストグラム）

In [ ]:
def _hist(ax, data, title, xlabel, log_y=True):
    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]
    if data.size:
        ax.hist(data, bins=60)
    if log_y:
        ax.set_yscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("count")


for (acc, dataset_id), pdat in plot_store.items():
    # 1. 非ゼロ値のヒストグラム
    fig, ax = plt.subplots(figsize=(5, 3.2))
    _hist(ax, pdat["values"], f"{acc} nonzero .X values", "value")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{acc}_value_hist.png", dpi=100)
    plt.close(fig)

    # 2. row sum のヒストグラム
    fig, ax = plt.subplots(figsize=(5, 3.2))
    _hist(ax, pdat["row_sums"], f"{acc} row sums", "row sum (per cell)")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{acc}_row_sum_hist.png", dpi=100)
    plt.close(fig)

    # 3. n_genes_by_cell のヒストグラム
    fig, ax = plt.subplots(figsize=(5, 3.2))
    _hist(ax, pdat["n_genes"], f"{acc} n_genes_by_cell", "n genes (>0) per cell", log_y=False)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{acc}_n_genes_hist.png", dpi=100)
    plt.close(fig)

print(f"saved plots -> {PLOT_DIR}")

## summary CSV を保存して主要列を表示

In [ ]:
summary_path = OUT_DIR / "preprocessing_state_summary.csv"
summary.to_csv(summary_path, index=False)
print("saved:", summary_path)

display_cols = [
    "source_accession", "dataset_id", "n_cells", "n_genes",
    "min", "max", "p99",
    "fraction_negative", "fraction_integer_like",
    "row_sum_median", "row_sum_cv",
    "guess_preprocessing", "confidence", "include_for_count_merge", "notes",
]
summary[[c for c in display_cols if c in summary.columns]]

## 次のステップ（04 へ）

- この 03 では QC フィルタリングは行っていない。**QC 本番は 04 で merge 対象を決めた後**。
- `include_for_count_merge == True`（= `raw_count_like`）のデータだけが、通常の
  `total_counts` / `n_genes_by_counts` / `pct_counts_mt` ベースの QC にそのまま適している。
- `log_normalized_like` や `scaled_zscore_like` のデータでは count-based QC の解釈が変わるため、
  そのまま混ぜず、04 で扱いを決める。
- `processed_count_like` は初期値 `include_for_count_merge=False`。SoupX 補正カウントなどは
  人間が中身を見て判断する。

次: **04_merge_curated_h5ad.ipynb**